## Load and Check data

In [2]:
import pandas as pd

# Load fungi biodiversity data frames
df_forest = pd.read_csv("data_raw/fungi_forest_2021_raw.csv")
df_grassland = pd.read_csv("data_raw/fungi_grassland_2021_raw.csv")

# Stack them vertically into one table and remove duplicates
# Ignore_index = True to remove the original row numbers and create a new index
df_fungi_div = pd.concat([df_forest, df_grassland], ignore_index=True)
df_fungi_div = df_fungi_div.drop_duplicates()

# Inspect merged table 
print(df_fungi_div.info())
print(f"\nUnique plots: {df_fungi_div['Plotid'].nunique()}")
print(f"Unique Fungal Species (OTUs): {df_fungi_div['OTU'].nunique()}")
df_fungi_div.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 47701 entries, 0 to 47700
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Plotid     47701 non-null  str  
 1   OTU        47701 non-null  str  
 2   Abundance  47701 non-null  int64
dtypes: int64(1), str(2)
memory usage: 1.1 MB
None

Unique plots: 300
Unique Fungal Species (OTUs): 6036


,Plotid,OTU,Abundance
0,AEW1,OTU0003,74
1,AEW1,OTU0006,630
2,AEW1,OTU0008,126


In [ ]:
# Load fungi taxonomy data frames, remove duplicates,and check 
df_fungi_tax = pd.read_csv("data_raw/fungi_tax.csv")
df_fungi_tax = df_fungi_tax.drop_duplicates()
print(df_fungi_tax.info())

print(f"Unique Fungal Species (OTUs): {df_fungi_div['OTU'].nunique()}")
df_fungi_tax.head(3)

Fungi DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 9784 entries, 0 to 9783
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   OTU                  9784 non-null   str  
 1   RepSeq               9784 non-null   str  
 2   Kingdom              9784 non-null   str  
 3   Phylum               9784 non-null   str  
 4   Class                9784 non-null   str  
 5   Order                9784 non-null   str  
 6   Family               9784 non-null   str  
 7   Genus                9784 non-null   str  
 8   Species              9784 non-null   str  
 9   primary_lifestyle    9784 non-null   str  
 10  secondary_lifestyle  9784 non-null   str  
 11  Guild                9784 non-null   str  
dtypes: str(12)
memory usage: 917.4 KB
None


,OTU,RepSeq,Kingdom,Phylum,Class,Order,Family,Genus,Species,primary_lifestyle,secondary_lifestyle,Guild
0,OTU0001,TTAAGTTCAGCGGGTATCCCTACCTGATCCGAGGTCAACCTTAGAA...,Fungi,Ascomycota,Eurotiomycetes,Chaetothyriales,Herpotrichiellaceae,Exophiala,unclassified,animal_parasite,litter_saprotroph,saprotroph
1,OTU0002,TTAAGTTCAGCGGGTATCCCTACCTGATCCGAGGTCAAATTCATGG...,Fungi,Ascomycota,Dothideomycetes,Pleosporales,Sporormiaceae,Preussia,unclassified,dung_saprotroph,unknown,saprotroph
2,OTU0003,TTAAGTTCAGCGGGTAGTCTTACTTGATTTGAGATCGAGTTGAACA...,Fungi,Mortierellomycota,Mortierellomycetes,Mortierellales,Mortierellaceae,Podila,Podila_humilis,unknown,unknown,unknown


In [11]:
# Load soil data frames
df_soil = pd.read_csv("data_raw/soil_chemistry_2021_raw.csv")

# Check the soil chemistry dataset
print(f"Soil DataFrame Info:")
print(df_soil.info())
# to list all columns: print("Soil Table Columns:", df_soil.columns.tolist())

print(f"\nUnique plots in soil dataset: {df_soil['Plotid'].nunique()}")
df_soil.head(3)

Soil DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   EP_Plotid    300 non-null    str    
 1   Plotid       300 non-null    str    
 2   Exploratory  300 non-null    str    
 3   Type         300 non-null    str    
 4   Total_C      300 non-null    float64
 5   Inorganic_C  300 non-null    float64
 6   Organic_C    300 non-null    float64
 7   Total_N      300 non-null    float64
 8   Total_S      300 non-null    float64
 9   CN_ratio     300 non-null    float64
 10  CS_ratio     300 non-null    float64
dtypes: float64(7), str(4)
memory usage: 25.9 KB
None

Unique plots in soil dataset: 300


,EP_Plotid,Plotid,Exploratory,Type,Total_C,Inorganic_C,Organic_C,Total_N,Total_S,CN_ratio,CS_ratio
0,AEG1,A19557,A,G,90.19,1.82,88.38,9.14,1.18,9.67,75.15
1,AEG2,A39275,A,G,85.56,2.30,83.26,8.87,1.18,9.39,70.44
2,AEG3,A48112,A,G,62.47,0.38,62.09,5.92,0.78,10.49,79.66


### Create Database and combine tables with SQLite 

In [ ]:
import sqlite3

# Connect to SQLite and create a database
conn = sqlite3.connect("biodiversity.db")

# Save datasets as tables
# index = False so the row numbers aren't saved as a column, and if it exists, overwrite it.
df_fungi_div.to_sql("fungi_div", conn, if_exists="replace", index=False)
df_fungi_tax.to_sql("fungi_tax", conn, if_exists="replace", index=False)
df_soil.to_sql("soil_chem", conn, if_exists="replace", index=False)

# 4. Commit changes and close connection
conn.commit()
conn.close()

Create a new table by joining them via SQLite

In [13]:
# Connect to database
conn = sqlite3.connect("/home/sylas/Desktop/DSR/Project/biodiversity.db")

# Read table into DataFrame
df_soil_fungi_div = pd.read_sql_query("SELECT * FROM fungi_data_join_soil", conn)

# Export to CSV (index=False prevents adding an extra row-number column)
df_soil_fungi_div.to_csv("/home/sylas/Desktop/DSR/Project/data_process/output.csv", index=False)

conn.close()